# Go2 Course Homework: Teaching Colab Template

This notebook is a **teaching-first** wrapper around the readable course code.

It assumes that the readable homework package is stored in a course GitHub repo.
Unlike the original payload-based notebook, this version keeps the important
source files visible as normal `.py` / `.json` files.

Recommended lecture usage:
1. run setup
2. inspect the environment
3. show the key files
4. restore a checkpoint and render a demo
5. run the public benchmark

## 1. Configure repository URLs

Set `COURSE_REPO_URL` to the GitHub repository that contains this readable
homework package. The repo should contain:
- `train.py`
- `test_policy.py`
- `generate_public_rollout.py`
- `public_eval.py`
- `go2_pg_env/`
- `configs/course_config.json`

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

COURSE_REPO_URL = "https://github.com/WeijieLai1024/EEC289A_Robotics-Homework.git"
COURSE_REPO_DIR = Path("/content/go2_course_repo")

PLAYGROUND_REPO = "https://github.com/google-deepmind/mujoco_playground.git"
PLAYGROUND_REF = "dd38c285c6d54266287081e516109f0b15985818"

UNITREE_MUJOCO_REPO = "https://github.com/unitreerobotics/unitree_mujoco.git"
UNITREE_MUJOCO_REF = "1a37b051a10be723405b7ed6dc839361af036d88"

PLAYGROUND_DIR = Path("/content/mujoco_playground")
UNITREE_DIR = Path("/content/unitree_mujoco")

if "google.colab" in sys.modules:
    print("Running inside Colab.")
else:
    print("This notebook was designed for Colab, but local execution may also work.")

## 2. Install system packages and clone repositories

In [ ]:
!command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg)
!python -m pip install -q -U pip setuptools wheel
!python -m pip uninstall -y playground || true

if not PLAYGROUND_DIR.exists():
    !git clone {PLAYGROUND_REPO} {PLAYGROUND_DIR}
!git -C {PLAYGROUND_DIR} fetch --all --tags
!git -C {PLAYGROUND_DIR} checkout {PLAYGROUND_REF}

if not UNITREE_DIR.exists():
    !git clone {UNITREE_MUJOCO_REPO} {UNITREE_DIR}
!git -C {UNITREE_DIR} fetch --all --tags
!git -C {UNITREE_DIR} checkout {UNITREE_MUJOCO_REF}

if not COURSE_REPO_DIR.exists():
    !git clone {COURSE_REPO_URL} {COURSE_REPO_DIR}

!python -m pip install -q -r {COURSE_REPO_DIR / 'configs' / 'colab_requirements.txt'}
%cd {PLAYGROUND_DIR}
!python -m pip install -q -e .
%cd {COURSE_REPO_DIR}

import jax
import mujoco_playground

print("JAX devices:", jax.devices())
print("JAX backend:", jax.default_backend())
print("mujoco_playground imported from:", mujoco_playground.__file__)
if "/content/mujoco_playground/" not in mujoco_playground.__file__:
    raise RuntimeError("Expected mujoco_playground to be imported from /content/mujoco_playground")

## 3. Copy Go2 assets from `unitree_mujoco` into the local course environment

In [ ]:
%cd {COURSE_REPO_DIR}
!python scripts/copy_go2_assets.py --unitree-dir {UNITREE_DIR} --course-dir {COURSE_REPO_DIR}

## 4. Inspect the environment before training

In [ ]:
%cd {COURSE_REPO_DIR}
!python inspect_env.py --stage-name stage_2

## 5. Read the most important files

In [ ]:
!sed -n '1,220p' go2_pg_env/joystick.py

In [ ]:
!sed -n '1,220p' train.py

In [ ]:
!sed -n '1,220p' benchmark_specs.py

## 6. Define a Colab-friendly runtime config

In [ ]:
import json

runtime_config = {
    "num_envs": 1024,
    "num_eval_envs": 128,
    "num_evals": 5,
    "batch_size": 256,
    "policy_hidden_layer_sizes": [256, 256, 128],
    "value_hidden_layer_sizes": [256, 256, 128],
    "stage_1_num_timesteps": 10_000_000,
    "stage_2_num_timesteps": 5_000_000,
}

config_path = COURSE_REPO_DIR / "configs" / "colab_runtime_config.json"
base_config_path = COURSE_REPO_DIR / "configs" / "course_config.json"
base_config = json.loads(base_config_path.read_text())
base_config["runtime_overrides"] = runtime_config
config_path.write_text(json.dumps(base_config, indent=2))
print("wrote", config_path)

## 7. Dry-run the training config

In [ ]:
!python train.py --config configs/colab_runtime_config.json --dry-run

## 8. Run training

For lecture use, it is usually better to **skip this cell live** and instead use
a staff checkpoint. Full training includes JIT compile time.

In [ ]:
# !python train.py #   --config configs/colab_runtime_config.json #   --stage both #   --output-dir artifacts/run_baseline

## 9. Restore a checkpoint and render a deterministic demo

Replace the checkpoint path below with a real trained checkpoint if needed.

In [ ]:
CHECKPOINT_DIR = COURSE_REPO_DIR / "artifacts" / "run_baseline" / "best_checkpoint"
DEMO_DIR = COURSE_REPO_DIR / "artifacts" / "demo_bundle"

# !python test_policy.py #   --config configs/colab_runtime_config.json #   --checkpoint-dir {CHECKPOINT_DIR} #   --stage-name stage_2 #   --output-dir {DEMO_DIR}

## 10. Generate the public benchmark rollout

In [ ]:
PUBLIC_DIR = COURSE_REPO_DIR / "artifacts" / "public_eval_bundle"

# !python generate_public_rollout.py #   --config configs/colab_runtime_config.json #   --checkpoint-dir {CHECKPOINT_DIR} #   --stage-name stage_2 #   --output-dir {PUBLIC_DIR} #   --num-episodes 4 #   --render-first-episode

# !python public_eval.py #   --config configs/colab_runtime_config.json #   --rollout-npz {PUBLIC_DIR / "rollout_public_eval.npz"} #   --output-json {PUBLIC_DIR / "public_eval.json"}

## 11. What to show in lecture

Recommended order:
1. `inspect_env.py`
2. `go2_pg_env/joystick.py`
3. `configs/course_config.json`
4. `test_policy.py`
5. `generate_public_rollout.py`
6. `public_eval.json`

This order mirrors the teaching goal:
**task -> train -> restore -> benchmark**